In [1]:
import dateutil.parser
import pickle
import os
import time
import pandas as pd
import numpy as np
import csv
from operator import itemgetter
from tqdm import tqdm

In [2]:
runtime = time.time()
MovieLens_1M = "MovieLens-1M"
MovieLens_20M = "MovieLens-20M"
steam = "steam"

In [3]:
# Uncomment the dataset you want to use here
dataset = steam

# Here you can change the path to the dataset
DATASET_DIR = './datasets/'+ dataset

In [4]:
def load_pickle(pickle_file):
    return pickle.load(open(pickle_file, 'rb'))

In [5]:
if dataset == MovieLens_1M:
    DATASET_FILE = DATASET_DIR + '/ratings.csv' #ml

elif dataset == MovieLens_20M:
    DATASET_FILE = DATASET_DIR + '/ratings.csv' #ml

elif dataset == steam:
    DATASET_FILE = DATASET_DIR + '/steam_reviews.json' #steam


DATASET_W_CONVERTED_TIMESTAMPS = DATASET_DIR + '/1_converted_timestamps.pickle'
DATASET_USER_ARTIST_MAPPED = DATASET_DIR + '/2_user_artist_mapped.pickle'
DATASET_USER_SESSIONS_1 = DATASET_DIR  + '/3_user_sessions_1.pickle'
DATASET_USER_SESSIONS_2 = DATASET_DIR  + '/3_user_sessions_2.pickle'
DATASET_TRAIN_TEST_SPLIT = DATASET_DIR + '/4_train_test_split.pickle'
DATASET_TRAIN_TEST_SPLIT_2 = DATASET_DIR + '/4_train_test_split_sapmle.pickle' #樣本的名稱
DATASET_BPR_MF = DATASET_DIR + '/bpr-mf_train_test_split.pickle'

if dataset == MovieLens_1M:
    SESSION_TIMEDELTA = 60*60*24 #1 hours 60*30
elif dataset == MovieLens_20M:
    SESSION_TIMEDELTA = 60*60*24 #1hours 60*30
elif dataset == steam:
    SESSION_TIMEDELTA = 60*60*24 #1 hours 60*30

MAX_SESSION_LENGTH = 15   #20  #fix # maximum number of events in a session
MINIMUM_REQUIRED_SESSIONS = 3 # The dual-RNN should have minimum 2 two train + 1 to test
PAD_VALUE = 0


SEED = 12345

MAX_SEQUENCE_LENGTH = 200 #fix
MIN_SESSION_COUNT_PER_USER = 2 #fix
MIN_ITEM_COUNT_PER_SESSION = 2 #fix
MIN_ITEM_COUNT_PER_USER = 5 #fix
MIN_USER_COUNT_PER_ITEM = 5 #fix
SESSION_WINDOW = 60*60*24 #fix
NUM_NEGATIVE_SAMPLES = 100 #?
NEGATIVE_SAMPLER_SEED = SEED #?

In [6]:
def file_exists(filename):
    return os.path.isfile(filename)

def load_pickle(pickle_file):
    return pickle.load(open(pickle_file, 'rb'))

def save_pickle(data_object, data_file):
    pickle.dump(data_object, open(data_file, 'wb'))

In [7]:
def convert_timestamps():
    dataset_list = []
    with open(DATASET_FILE, 'rt', buffering=10000, encoding='utf8') as datasets:
        for line in datasets:
            if dataset == MovieLens_1M:
                line = line.rstrip()
                line = line.split(':')
                if line[6] == 'timestamp':
                    continue
                user_id   = line[0]
                item      = line[2]
                timestamp = float(line[6])
                dataset_list.append( [user_id, timestamp, item] )

            elif dataset == MovieLens_20M:
                line = line.rstrip()
                line = line.split(',')
                if line[3] == 'timestamp':
                    continue
                user_id   = line[0] 
                item      = line[1]
                timestamp = float(line[3])
                dataset_list.append( [user_id, timestamp, item] )

            elif dataset == steam:
                trans = eval(line)
                user_id    = trans['username']
                item       = trans['product_id']
                timestamp  = trans['date']
                struct_time = time.strptime(timestamp, "%Y-%m-%d") # 轉成時間元組
                timestamp = int(time.mktime(struct_time)) # 轉成時間戳
                dataset_list.append( [user_id, timestamp, item] )

    #dataset_list = list(reversed(dataset_list))
    save_pickle(dataset_list, DATASET_W_CONVERTED_TIMESTAMPS)

In [8]:
def map_user_and_artist_id_to_labels():
    dataset_list = load_pickle(DATASET_W_CONVERTED_TIMESTAMPS)
    artist_map = {}
    user_map = {}
    artist_id = ''
    user_id = ''
    for i in range(len(dataset_list)):
        user_id = dataset_list[i][0]
        artist_id = dataset_list[i][2]
        
        if user_id not in user_map:
            user_map[user_id] = len(user_map)
        if artist_id not in artist_map:
            artist_map[artist_id] = len(artist_map)
        
        dataset_list[i][0] = user_map[user_id]
        dataset_list[i][2] = artist_map[artist_id]
    
    # list to dataframe 
    dataset_list = pd.DataFrame(dataset_list) 
    # Save to pickle file
    save_pickle(dataset_list, DATASET_USER_ARTIST_MAPPED)

In [9]:
def reduce_ineligible_interactions_and_history():
    start_time = time.time()
    
    # load data
    print("- load data")
    dataset = load_pickle(DATASET_USER_ARTIST_MAPPED)
    update_dataset = dataset
    update_dataset.columns = ['User_ID', 'timestamp', 'Item_ID']
    
    # filter out tiny items
    print("- filter out tiny items")
    df_iid2ucount = update_dataset.groupby('Item_ID').size()
    survived_iids = df_iid2ucount.index[df_iid2ucount >= MIN_USER_COUNT_PER_ITEM]
    update_dataset = update_dataset[update_dataset['Item_ID'].isin(survived_iids)]
    
    # filter out tiny users
    print("- filter out tiny users")
    df_uid2icount = update_dataset.groupby('User_ID').size()
    survived_uids = df_uid2icount.index[df_uid2icount >= MIN_ITEM_COUNT_PER_USER]
    update_dataset = update_dataset[update_dataset['User_ID'].isin(survived_uids)]

    
    end_time = (time.time() - start_time)/60
    end_time = "%.2f" % end_time
    print("Complete | Before: "+str(len(dataset))+" pieces of data | After: "+str(len(update_dataset))+" pieces of data | Runtime Spend: "+str(end_time)+" min")

    save_pickle(update_dataset, DATASET_USER_SESSIONS_1)

In [10]:
def cut_and_assign_sids_to_rows(rows):
    sid = 0
    uid2rows = {}
    for uid, timestamp, iid in tqdm(rows, desc="* organize uid2rows"):
        if uid not in uid2rows:
            uid2rows[uid] = []
        uid2rows[uid].append((iid, timestamp))
    rows = []
    uids = list(uid2rows.keys())
    for uid in tqdm(uids, desc="* cutting"):
        user_rows = sorted(uid2rows[uid], key=itemgetter(1))
        tba = []
        sid2count = {}
        #'''
        if MAX_SEQUENCE_LENGTH:
            user_rows = user_rows[-MAX_SEQUENCE_LENGTH:]
            #user_rows = user_rows
        #'''
        sid += 1
        _, previous_timestamp = user_rows[0]
        for iid, timestamp in user_rows:
            if timestamp - previous_timestamp > SESSION_WINDOW:
                sid += 1
            tba.append((uid, timestamp, iid))
            sid2count[sid] = sid2count.get(sid, 0) + 1
            previous_timestamp = timestamp
        if MIN_SESSION_COUNT_PER_USER and len(sid2count) < MIN_SESSION_COUNT_PER_USER:
            continue
        if MIN_ITEM_COUNT_PER_SESSION and min(sid2count.values()) < MIN_ITEM_COUNT_PER_SESSION:
            continue
        rows.extend(tba)
    return rows

In [11]:
def split_single_session(session):
    splitted = [session[i:i+MAX_SESSION_LENGTH] for i in range(0, len(session), MAX_SESSION_LENGTH)]
    
    ''' #將這個註解 可以和BERT4Rec處理數據一致
    if len(splitted[-1]) < 2: #長度小於多少就刪除 需要考慮 訓練 測試集問題
        del splitted[-1]

    '''
        
    return splitted

In [12]:
def perform_session_splits(sessions):
    splitted_sessions = []
    for session in sessions:
        splitted_sessions += split_single_session(session)

    return splitted_sessions

In [13]:
def split_long_sessions(user_sessions):
    for k, v in user_sessions.items():
        user_sessions[k] = perform_session_splits(v)

In [14]:
def sort_and_split_usersessions():
    #start_time = time.time()
    
    # load data
    print("- load data")
    dataset = load_pickle(DATASET_USER_SESSIONS_1)
    
    # cut and assign sid
    print("- cut and assign sid")
    rows = cut_and_assign_sids_to_rows(dataset.values)
    dataset = pd.DataFrame(rows)
    dataset.columns = ['User_ID', 'timestamp', 'Item_ID']
    
    # map uid -> uindex
    print("- map uid -> uindex")
    uids = set(dataset['User_ID'])
    uid2uindex = {uid: index for index, uid in enumerate(set(uids), start=1)}
    dataset['uindex'] = dataset['User_ID'].map(uid2uindex)
    dataset = dataset.drop(columns=['User_ID'])
    with open(os.path.join(DATASET_DIR, 'uid2uindex.pkl'), 'wb') as fp:
        pickle.dump(uid2uindex, fp)
        
    # map iid -> iindex
    print("- map iid -> iindex")
    iids = set(dataset['Item_ID'])
    iid2iindex = {iid: index for index, iid in enumerate(set(iids), start=1)}
    dataset['iindex'] = dataset['Item_ID'].map(iid2iindex)
    dataset = dataset.drop(columns=['Item_ID'])
    with open(os.path.join(DATASET_DIR, 'iid2iindex.pkl'), 'wb') as fp:
        pickle.dump(iid2iindex, fp)

    #######################################
    # df to lsit
    time = list(dataset['timestamp'])
    user = list(dataset['uindex'])
    item = list(dataset['iindex'])

    temp_datalist = []
    for i in range(len(dataset)):
        temp_datalist.append([user[i], time[i], item[i]])

    # list to dict
    user_sessions = {}
    current_session = []
    for event in temp_datalist:
        user_id = event[0]
        timestamp = event[1]
        artist = event[2]

        new_event = [timestamp, artist]

        # if new user -> new session
        if user_id not in user_sessions:
            user_sessions[user_id] = []
            current_session = []
            user_sessions[user_id].append(current_session)
            current_session.append(new_event)
            continue

        # it is an existing user: is it a new session?
        # we also know that the current session contains at least one event
        # NB: Dataset is presorted from newest to oldest events
        last_event = current_session[-1]
        last_timestamp = last_event[0]
        timedelta = timestamp - last_timestamp

        if timedelta < SESSION_TIMEDELTA:
            # new event belongs to current session
            current_session.append(new_event)
        else:
            # new event belongs to new session
            current_session = [new_event]
            user_sessions[user_id].append(current_session)
    
    split_long_sessions(user_sessions)
    
    #end_time = (time.time() - start_time)/60
    #end_time = "%.2f" % end_time
    #print("Runtime Spend: "+str(end_time)+" min")
    
    save_pickle(user_sessions, DATASET_USER_SESSIONS_2)

In [15]:
'''
dataset = load_pickle(DATASET_USER_SESSIONS_2)
trainset = {}
testset = {}

for k, v in dataset.items():
    if k == 500:   
        print(k)
        print(v)
        print()
        trainset[k] = v[:-1]
        testset[k] = v[-1:]
        print(trainset[k])
        print(testset[k])
        print()
'''

'\ndataset = load_pickle(DATASET_USER_SESSIONS_2)\ntrainset = {}\ntestset = {}\n\nfor k, v in dataset.items():\n    if k == 500:   \n        print(k)\n        print(v)\n        print()\n        trainset[k] = v[:-1]\n        testset[k] = v[-1:]\n        print(trainset[k])\n        print(testset[k])\n        print()\n'

In [16]:
def get_session_lengths(dataset):
    session_lengths = {}
    for k, v in dataset.items():
        session_lengths[k] = []
        for session in v:
            session_lengths[k].append(len(session)-1)

    return session_lengths

In [17]:
def create_padded_sequence(session):
    if len(session) == MAX_SESSION_LENGTH:
        return session

    dummy_timestamp = 0
    dummy_label = 0
    length_to_pad = MAX_SESSION_LENGTH - len(session)
    padding = [[dummy_timestamp, dummy_label]] * length_to_pad
    session += padding
    #session.append(padding)
    return session

In [18]:
def pad_sequences(dataset):
    for k, v in dataset.items():
        #dataset[k] = create_padded_sequence(dataset[k])
        for session_index in range(len(v)):
            dataset[k][session_index] = create_padded_sequence(dataset[k][session_index])

# Splits the dataset into a training and a testing set, by extracting the last
# sessions of each user into the test set

In [19]:
def split_to_training_and_testing():
    dataset = load_pickle(DATASET_USER_SESSIONS_2)
    trainset = {}
    testset = {}
    
    for k, v in dataset.items():
        n_sessions = len(v)
        split_point = int(0.8*n_sessions)
        
        # runtime check to ensure that we have enough sessions for training and testing
        if split_point < 1: #2
            raise ValueError('User '+str(k)+' with '+str(n_sessions)+""" sessions, 
                resulted in split_point: '+str(split_point)+' which gives too 
                few training sessions. Please check that data and preprocessing 
                is correct.""")
        
        trainset[k] = v[:split_point]
        testset[k] = v[split_point:]
        
    '''
    for k, v in dataset.items():

        trainset[k] = v[:-1]
        testset[k] = v[-1:]
   '''
    
    # Also need to know session lengths for train- and testset
    train_session_lengths = get_session_lengths(trainset)
    test_session_lengths = get_session_lengths(testset)

    # Finally, pad all sequences before storing everything
    pad_sequences(trainset)
    pad_sequences(testset)
    

    # Put everything we want to store in a dict, and just store the dict with pickle
    pickle_dict = {}
    pickle_dict['trainset'] = trainset
    pickle_dict['testset'] = testset
    pickle_dict['train_session_lengths'] = train_session_lengths
    pickle_dict['test_session_lengths'] = test_session_lengths
    
    save_pickle(pickle_dict , DATASET_TRAIN_TEST_SPLIT)

In [20]:
def split_to_training_and_testing_sample():
    
    #切樣本的地方從這開始
    #單純用計數器，到達檻時就break掉
    dataset = load_pickle(DATASET_USER_SESSIONS_2)
    count = 0
    split_sample = {}

    for k, v in dataset.items():
        split_sample[k] = v

        count += 1
        if count == 100:   #取到100位user(但會算0，所以key只有到99)
            break
    
    trainset = {}
    testset = {}
    new_input_trainset = {}
    new_input_testset = {}

    for k, v in split_sample.items():
        n_sessions = len(v)
        split_point = int(0.8*n_sessions)
        
        # runtime check to ensure that we have enough sessions for training and testing
        if split_point < 1: #2
            raise ValueError('User '+str(k)+' with '+str(n_sessions)+""" sessions, 
                resulted in split_point: '+str(split_point)+' which gives too 
                few training sessions. Please check that data and preprocessing 
                is correct.""")
        
        trainset[k] = v[:split_point]
        testset[k] = v[split_point:]

    
    '''
    for k, v in split_sample.items():

        trainset[k] = v[:-1]
        testset[k] = v[-1:]
    
    '''
    
    # Also need to know session lengths for train- and testset
    train_session_lengths = get_session_lengths(trainset)
    test_session_lengths = get_session_lengths(testset)

    # Finally, pad all sequences before storing everything
    pad_sequences(trainset)
    pad_sequences(testset)
        

    # Put everything we want to store in a dict, and just store the dict with pickle
    pickle_dict = {}
    pickle_dict['trainset'] = trainset
    pickle_dict['testset'] = testset
    pickle_dict['train_session_lengths'] = train_session_lengths
    pickle_dict['test_session_lengths'] = test_session_lengths
    
    save_pickle(pickle_dict , DATASET_TRAIN_TEST_SPLIT_2)

In [21]:
def create_bpr_mf_sets():
    p = load_pickle(DATASET_TRAIN_TEST_SPLIT)
    train = p['trainset']
    train_sl = p['train_session_lengths']
    test = p['testset']
    test_sl = p['test_session_lengths']

    for user in train.keys():
        extension = test[user][:-1]
        train[user].extend(extension)
        extension = test_sl[user][:-1]
        train_sl[user].extend(extension)
    
    for user in test.keys():
        test[user] = [test[user][-1]]
        test_sl[user] = [test_sl[user][-1]]

    pickle_dict = {}
    pickle_dict['trainset'] = train
    pickle_dict['testset'] = test
    pickle_dict['train_session_lengths'] = train_sl
    pickle_dict['test_session_lengths'] = test_sl
    
    save_pickle(pickle_dict , DATASET_BPR_MF)

In [22]:
if not file_exists(DATASET_W_CONVERTED_TIMESTAMPS):
    print("Converting timestamps.")
    convert_timestamps()
    print("--------------------------------------")
    
if not file_exists(DATASET_USER_ARTIST_MAPPED):
    print("Mapping user and artist IDs to labels.")
    map_user_and_artist_id_to_labels()
    print("--------------------------------------")

if not file_exists(DATASET_USER_SESSIONS_1):
    print("Reduce_ineligible_interactions_and_history.")
    reduce_ineligible_interactions_and_history()
    print("--------------------------------------")
    
if not file_exists(DATASET_USER_SESSIONS_2):
    print("Sorting sessions to users.")
    sort_and_split_usersessions()
    print("--------------------------------------")
    
if not file_exists(DATASET_TRAIN_TEST_SPLIT):
    print("Splitting dataset into training and testing sets.")
    split_to_training_and_testing()
    print("--------------------------------------")
    
if not file_exists(DATASET_TRAIN_TEST_SPLIT_2):
    print("Splitting dataset into training and testing sets to Sample Data.")
    split_to_training_and_testing_sample()
    print("--------------------------------------")
    
if not file_exists(DATASET_BPR_MF):
    print("Creating dataset for BPR-MF.")
    create_bpr_mf_sets()
    print("--------------------------------------")
    
end_time = (time.time() - runtime)/60
end_time = "%.2f" % end_time
print("Runtime Spend:", str(end_time), "min")

Converting timestamps.
--------------------------------------
Mapping user and artist IDs to labels.
--------------------------------------
Reduce_ineligible_interactions_and_history.
- load data
- filter out tiny items
- filter out tiny users
Complete | Before: 7793069 pieces of data | After: 4212380 pieces of data | Runtime Spend: 0.05 min
--------------------------------------
Sorting sessions to users.
- load data
- cut and assign sid


* cutting: 100%|████████████████████████████████████████████████████████████| 334542/334542 [00:04<00:00, 72951.37it/s]


- map uid -> uindex
- map iid -> iindex
--------------------------------------
Splitting dataset into training and testing sets.
--------------------------------------
Splitting dataset into training and testing sets to Sample Data.
--------------------------------------
Creating dataset for BPR-MF.
--------------------------------------
Runtime Spend: 7.42 min
